# Shakespeare task

In [ ]:
# Import required libraries
import numpy as np
import requests
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# URL of the Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Fetching the text data from the URL
response = requests.get(url)
text = response.text

# Printing the length of the fetched text
print("Length of text: ", len(text))

C:\Users\nazne\AppData\Roaming\Python\Python314\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Length of text:  1115394


In [ ]:
chars = sorted(list(set(text)))
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

data = [char_to_ix[ch] for ch in text]

print(f"Unique characters: {len(chars)}")
print(f"Example mapping: {char_to_ix}")

Unique characters: 65
Example mapping: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


The above code turns each character in the text into a number that a model can understand. It also keeps a map to convert numbers back to characters.

**Changes made**
- Removed embedding layer and updated input size to vocab_size
- The model now expects one-hot encoded vectors instead of indices


In [ ]:
# Building the model
class RNNModel(nn.Module):
   def __init__(self, vocab_size, hidden_size, num_layers):
       super(RNNModel, self).__init__()
       self.rnn = nn.RNN(vocab_size, hidden_size, num_layers, batch_first=True)
       self.fc = nn.Linear(hidden_size, vocab_size)


   def forward(self, x, hidden):
       out, hidden = self.rnn(x, hidden)
       out = out.reshape(-1, out.shape[2])
       # generates predictions
       out = self.fc(out)
       return out, hidden

In [ ]:
# Creates mini-batches that help with data training
# set sequence length to 100

seq_length = 100
def get_batches(data, batch_size):
    n_batches = len(data) // (batch_size * seq_length)
    data = data[:n_batches * batch_size * seq_length]
    x = np.array(data)
    y = np.roll(x, -1)
    x = x.reshape(batch_size, -1)
    y = y.reshape(batch_size, -1)
    return x, y

In [ ]:
# number of sequences in a batch
batch_size = 64
x, y = get_batches(data, batch_size)
# displays data shapes
print(f"Input shape: {x.shape}, target shape: {y.shape}")

Input shape: (64, 17400), target shape: (64, 17400)


**Changes made**
- Added one-hot encoding for inputs and targets
- Converts sequences to binary

In [ ]:
# training the RNN Model
def train_model(model, data, epochs, batch_size, seq_length, vocab_size, lr = 0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        x, y = get_batches(data, batch_size)
        hidden = None

        for i in range(0, x.shape[1], seq_length):
            inputs = torch.tensor(x[:, i:i+seq_length], dtype=torch.long)
            targets = torch.tensor(y[:, i:i+seq_length], dtype=torch.long)

            # added one-hot encode inputs
            inputs = F.one_hot(inputs, num_classes=vocab_size).float()

            # created one-hot encoded targets
            targets_one_hot = F.one_hot(targets, num_classes=vocab_size).float()

            # clear gradients
            optimizer.zero_grad()

            # detach hidden state to prevent graph build up
            if hidden is not None:
                hidden = hidden.detach()

            outputs, hidden = model(inputs, hidden)

            targets = targets.reshape(-1)

            # compute loss
            loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))

            loss.backward()
            optimizer.step()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# hyperparameter for our RNN model
vocab_size = len(chars)
hidden_size = 256
num_layers = 2

In [ ]:
# Initialize and train the model
model = RNNModel(vocab_size, hidden_size, num_layers)
train_model(model, data, epochs=10, batch_size=64, seq_length=100, vocab_size=vocab_size)

Epoch [1/10], Loss: 2.3425
Epoch [2/10], Loss: 2.0553
Epoch [3/10], Loss: 1.8941
Epoch [4/10], Loss: 1.7879
Epoch [5/10], Loss: 1.7095
Epoch [6/10], Loss: 1.6560
Epoch [7/10], Loss: 1.6101
Epoch [8/10], Loss: 1.5740
Epoch [9/10], Loss: 1.5456
Epoch [10/10], Loss: 1.5197


The model was trained for 10 epochs using character-level RNN. The training loss decreased steadily from 2.3425 in the first epoch to 1.5197 in the final epoch. The constant reduction shows that the model is continuously learning. As the loss decreases the model becomes better at predicting the next character in the sequence. 

Even though the model is able to predict some of the text, the generated text might still appear random or lack logic. This could be due to the following factors:

- The model is too simple
- Training was limited to 10 epochs
- This type of dataset with character-levels require more training



**Changes made**
- Updated text generation to stop after a 100 words instead of 100 characters
- Use split() to count words in the generate text

In [ ]:
def generate_text(model, seed, hidden, vocab_size, max_words=100):
    model.eval()
    input = torch.tensor([char_to_ix[seed]], dtype=torch.long).unsqueeze(0)
    # one-hot encode each new input character
    input = F.one_hot(input, num_classes=vocab_size).float()

    generated = seed

    while len(generated.split()) < max_words:
        outputs, hidden = model(input, hidden)

        prob = F.softmax(outputs[-1], dim=-1).data
        char_ix = torch.multinomial(prob, 1).item()

        next_char = ix_to_char[char_ix]
        generated += next_char

        input = torch.tensor([[char_ix]], dtype=torch.long)
        # added one-hot encoding here as well
        input = F.one_hot(input, num_classes=vocab_size).float()

    return generated

In [ ]:
generated_text = generate_text(model, seed="T", hidden=None, vocab_size=vocab_size, max_words=100)

print("Generated Text:\n")
print(generated_text)

Generated Text:

Thy vilent; hath will wonspinion!
If this pall; for a that you runy 'preation.
Ormer, if I most oath. Orle; our wandants,
God that far haum, there ih the with me; ly,
As his found aud patrowisib, as your hair,
Why, in?

Catenster I am we perity, you all;
Wheches like and good jest made my access,
Free foely with teram as child you An thy neburband,
Whether in hear in his most cat beauty.

BAPTISTA:
Betame thy larks, and buck; reskon uping to do,
Hevat one coadish marmia words, comently,
And no pray tree grace of Citizen the i
